# Large-Scale Web Log Analysis Project

## Part 1: Data Generation & Bronze Ingestion

### Project Overview
Analyzing clickstream logs from an e-commerce website.

**Analysis Goals:**
1. Funnel analysis (visit → product view → cart → purchase conversion rate)
2. User sessionization (session boundary at 30 min of inactivity)
3. Anomaly detection (bot traffic, abnormal patterns)
4. Traffic patterns by time of day / day of week
5. Behavioral data aggregation for product recommendations

**Data Volume:** ~5 million events (7 days of clickstream)

**Pipeline:**
```
Raw JSON → Bronze (Parquet) → Silver (sessionization, cleansing) → Gold (analytics metrics)
```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random
import json
import os
import shutil
import time

spark = SparkSession.builder \
    .appName('LogAnalysis-Part1-DataGen') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

sc = spark.sparkContext

# Project directories
PROJECT = '/home/jovyan/data/log_analysis'
RAW = f'{PROJECT}/raw'
LAKE = f'{PROJECT}/lake'

for d in [RAW, f'{LAKE}/bronze', f'{LAKE}/silver', f'{LAKE}/gold']:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print(f'✅ Spark UI: http://localhost:4040')
print(f'✅ Project: {PROJECT}')

✅ Spark UI: http://localhost:4040
✅ Project: /home/jovyan/data/log_analysis


---
## 1. Realistic Clickstream Data Generation

Reflects real web log patterns:
- Traffic variation by hour (low overnight, peaks at lunch/evening)
- User behavior patterns (following funnel order)
- Bot traffic (~5%)
- Variety of devices and browsers

In [2]:
random.seed(42)

# Configuration
NUM_USERS = 50_000
NUM_PRODUCTS = 2_000
NUM_DAYS = 7
START_DATE = datetime(2025, 6, 1)

# Hourly traffic weights (hours 0~23)
HOURLY_WEIGHTS = [
    0.1, 0.05, 0.03, 0.02, 0.02, 0.03,  # 0~5 (late night)
    0.05, 0.1, 0.2, 0.3, 0.4, 0.5,       # 6~11 (morning)
    0.6, 0.5, 0.4, 0.3, 0.3, 0.4,        # 12~17 (afternoon)
    0.5, 0.6, 0.7, 0.6, 0.4, 0.2,        # 18~23 (evening)
]

# Day-of-week weights (Mon~Sun)
DOW_WEIGHTS = [0.8, 0.8, 0.9, 0.9, 1.0, 1.2, 1.1]

# Event types and funnel conversion rates
EVENT_FUNNEL = {
    'page_view': 1.0,      # all sessions
    'product_view': 0.6,   # 60% view a product
    'add_to_cart': 0.25,   # 25% add to cart
    'checkout': 0.12,      # 12% start checkout
    'purchase': 0.08,      # 8% complete purchase
}

# Page list
PAGES = [
    '/', '/category/electronics', '/category/clothing', '/category/food',
    '/category/sports', '/category/books', '/search', '/cart', '/checkout',
    '/account', '/help', '/about'
]

# Devices and browsers
DEVICES = ['desktop', 'mobile', 'tablet']
DEVICE_WEIGHTS = [0.4, 0.5, 0.1]
BROWSERS = ['Chrome', 'Safari', 'Firefox', 'Edge', 'Samsung Internet']
OS_LIST = ['Windows', 'macOS', 'iOS', 'Android', 'Linux']
REFERRERS = ['google', 'naver', 'direct', 'instagram', 'facebook', 'kakaotalk', None]

print(f'Users: {NUM_USERS:,}')
print(f'Products: {NUM_PRODUCTS:,}')
print(f'Period: {START_DATE.strftime("%Y-%m-%d")} ~ {(START_DATE + timedelta(days=NUM_DAYS-1)).strftime("%Y-%m-%d")}')

유저 수: 50,000
상품 수: 2,000
기간: 2025-06-01 ~ 2025-06-07


In [3]:
def generate_user_session(user_id, session_start, is_bot=False):
    """Generate session events for a single user"""
    events = []
    current_time = session_start
    device = random.choices(DEVICES, weights=DEVICE_WEIGHTS)[0]
    browser = random.choice(BROWSERS)
    user_os = random.choice(OS_LIST)
    referrer = random.choice(REFERRERS)
    session_id = f's_{user_id}_{int(session_start.timestamp())}'
    
    if is_bot:
        # Bot: many page requests at high speed
        n_events = random.randint(50, 200)
        for i in range(n_events):
            events.append({
                'event_id': f'e_{user_id}_{int(current_time.timestamp())}_{i}',
                'user_id': f'bot_{user_id}',
                'session_id': session_id,
                'event_type': 'page_view',
                'page': random.choice(PAGES),
                'product_id': None,
                'timestamp': current_time.isoformat(),
                'device': 'desktop',
                'browser': 'Python-urllib',
                'os': 'Linux',
                'referrer': None,
                'response_time_ms': random.randint(5, 30),
                'status_code': 200,
            })
            current_time += timedelta(seconds=random.uniform(0.1, 2))
        return events
    
    # Regular user: funnel-based behavior
    # 1. Landing page
    events.append({
        'event_id': f'e_{user_id}_{int(current_time.timestamp())}_0',
        'user_id': f'u_{user_id}',
        'session_id': session_id,
        'event_type': 'page_view',
        'page': random.choice(PAGES[:7]),
        'product_id': None,
        'timestamp': current_time.isoformat(),
        'device': device,
        'browser': browser,
        'os': user_os,
        'referrer': referrer,
        'response_time_ms': random.randint(50, 500),
        'status_code': random.choices([200, 200, 200, 200, 301, 404, 500], k=1)[0],
    })
    current_time += timedelta(seconds=random.uniform(3, 30))
    
    # 2~5 additional page views
    n_pages = random.randint(1, 8)
    for i in range(n_pages):
        current_time += timedelta(seconds=random.uniform(5, 120))
        events.append({
            'event_id': f'e_{user_id}_{int(current_time.timestamp())}_{i+1}',
            'user_id': f'u_{user_id}',
            'session_id': session_id,
            'event_type': 'page_view',
            'page': random.choice(PAGES),
            'product_id': None,
            'timestamp': current_time.isoformat(),
            'device': device,
            'browser': browser,
            'os': user_os,
            'referrer': referrer,
            'response_time_ms': random.randint(50, 500),
            'status_code': 200,
        })
    
    # Funnel events (probability-based)
    funnel_events = ['product_view', 'add_to_cart', 'checkout', 'purchase']
    product_id = random.randint(1, NUM_PRODUCTS)
    
    for evt_type in funnel_events:
        if random.random() > EVENT_FUNNEL[evt_type]:
            break
        current_time += timedelta(seconds=random.uniform(10, 180))
        page = f'/product/{product_id}' if evt_type == 'product_view' else f'/{evt_type}'
        events.append({
            'event_id': f'e_{user_id}_{int(current_time.timestamp())}_{evt_type}',
            'user_id': f'u_{user_id}',
            'session_id': session_id,
            'event_type': evt_type,
            'page': page,
            'product_id': product_id,
            'timestamp': current_time.isoformat(),
            'device': device,
            'browser': browser,
            'os': user_os,
            'referrer': referrer,
            'response_time_ms': random.randint(80, 800),
            'status_code': 200,
        })
    
    return events

print('User session generator ready')

유저 세션 생성 함수 준비 완료


In [4]:
# Generate 7 days of data
total_events = 0

for day_offset in range(NUM_DAYS):
    date = START_DATE + timedelta(days=day_offset)
    date_str = date.strftime('%Y-%m-%d')
    dow = date.weekday()  # 0=Mon
    day_weight = DOW_WEIGHTS[dow]
    
    day_events = []
    
    # Generate sessions by hour
    for hour in range(24):
        hour_weight = HOURLY_WEIGHTS[hour] * day_weight
        n_sessions = int(3000 * hour_weight)  # sessions per hour
        
        for _ in range(n_sessions):
            user_id = random.randint(1, NUM_USERS)
            minute = random.randint(0, 59)
            second = random.randint(0, 59)
            session_start = date.replace(hour=hour, minute=minute, second=second)
            
            is_bot = random.random() < 0.05  # 5% bots
            events = generate_user_session(user_id, session_start, is_bot)
            day_events.extend(events)
    
    # Save daily file
    output_path = f'{RAW}/{date_str}.json'
    with open(output_path, 'w') as f:
        for evt in day_events:
            f.write(json.dumps(evt, ensure_ascii=False) + '\n')
    
    total_events += len(day_events)
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f'  {date_str} ({["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][dow]}): '
          f'{len(day_events):>8,} events, {size_mb:.1f} MB')

print(f'\nTotal events: {total_events:,}')
total_size = sum(os.path.getsize(f'{RAW}/{f}') for f in os.listdir(RAW)) / 1024 / 1024
print(f'Total size: {total_size:.1f} MB')

  2025-06-01 (일):  296,617건, 94.7 MB
  2025-06-02 (월):  224,533건, 71.7 MB
  2025-06-03 (화):  211,939건, 67.7 MB
  2025-06-04 (수):  236,782건, 75.6 MB
  2025-06-05 (목):  242,023건, 77.3 MB
  2025-06-06 (금):  269,772건, 86.1 MB
  2025-06-07 (토):  314,864건, 100.5 MB

총 이벤트: 1,796,530건
총 크기: 573.5 MB


---
## 2. Bronze Layer: Raw → Parquet Conversion

In [5]:
# Schema definition
log_schema = StructType([
    StructField('event_id', StringType()),
    StructField('user_id', StringType()),
    StructField('session_id', StringType()),
    StructField('event_type', StringType()),
    StructField('page', StringType()),
    StructField('product_id', IntegerType()),
    StructField('timestamp', TimestampType()),
    StructField('device', StringType()),
    StructField('browser', StringType()),
    StructField('os', StringType()),
    StructField('referrer', StringType()),
    StructField('response_time_ms', IntegerType()),
    StructField('status_code', IntegerType()),
])

# Read raw JSON
start = time.time()
raw_df = spark.read.schema(log_schema).json(f'{RAW}/')

# Bronze: add metadata + partitioning
bronze_df = raw_df \
    .withColumn('event_date', F.to_date('timestamp')) \
    .withColumn('event_hour', F.hour('timestamp')) \
    .withColumn('_ingested_at', F.current_timestamp())

bronze_df.write \
    .mode('overwrite') \
    .partitionBy('event_date') \
    .parquet(f'{LAKE}/bronze/clickstream')

elapsed = time.time() - start
count = bronze_df.count()

print(f'Bronze ingestion complete: {count:,} rows, {elapsed:.1f}s')

# Check Parquet size
bronze_path = f'{LAKE}/bronze/clickstream'
total_parquet = 0
for root, dirs, files in os.walk(bronze_path):
    for f in files:
        if f.endswith('.parquet'):
            total_parquet += os.path.getsize(os.path.join(root, f))
print(f'Parquet size: {total_parquet/1024/1024:.1f} MB ({total_size/(total_parquet/1024/1024):.1f}x compression vs JSON)')

Bronze 적재 완료: 1,796,530건, 5.1초
Parquet 크기: 38.9 MB (JSON 대비 14.7x 압축)


In [6]:
# Explore Bronze data
bronze = spark.read.parquet(f'{LAKE}/bronze/clickstream')

print('=== Schema ===')
bronze.printSchema()

print('\n=== Sample Data ===')
bronze.show(5, truncate=False)

print('\n=== Event Type Distribution ===')
bronze.groupBy('event_type').count().orderBy(F.col('count').desc()).show()

print('\n=== Daily Event Counts ===')
bronze.groupBy('event_date').count().orderBy('event_date').show()

print('\n=== Device Distribution ===')
bronze.groupBy('device').count().orderBy(F.col('count').desc()).show()

print('\n=== Bot vs Regular Users ===')
bronze.withColumn('is_bot', F.col('user_id').startswith('bot_')) \
    .groupBy('is_bot').agg(
        F.count('*').alias('events'),
        F.countDistinct('user_id').alias('users'),
        F.countDistinct('session_id').alias('sessions'),
    ).show()

=== 스키마 ===
root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- device: string (nullable = true)
 |-- browser: string (nullable = true)
 |-- os: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- event_date: date (nullable = true)


=== 샘플 데이터 ===
+--------------------+-------+------------------+----------+----------------+----------+--------------------------+-------+-------+-------+--------+----------------+-----------+----------+--------------------------+----------+
|event_id            |user_id|session_id        |event_type|page            |pr

In [7]:
spark.stop()
print('Part 1 complete: Data generation & Bronze ingestion')
print('Next: Part 2 - Silver (sessionization, cleansing, bot filtering)')

Part 1 완료: 데이터 생성 & Bronze 적재
다음: Part 2 - Silver (세션화, 정제, 봇 필터링)
